# Week 4: Pasqal Cloud — Wildfire Sensor Placement on Neutral Atom QPU

**Objective:** Submit wildfire sensor placement optimization to Pasqal Cloud for execution on real neutral atom hardware (Fresnel QPU) or emulators.

**Cloud Workflow:**
1. Authenticate with Pasqal Cloud SDK
2. Create Pulser sequence from sensor instance
3. Submit batch of jobs (multiple seeds/parameters)
4. Monitor job status and retrieve results
5. Compare cloud results to local simulation

**Available Devices:**
- `FRESNEL` — 100-qubit neutral atom QPU (hardware, requires quota)
- `EMU_FREE` — Free CPU emulator (up to 10 qubits)
- `EMU_TN` — Tensor network emulator (10-100 qubits, paid)
- `EMU_MPS` — Matrix product state emulator (GPU-accelerated, paid)

---

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter

# Pasqal Cloud SDK
try:
    from pasqal_cloud import SDK
    from pasqal_cloud.device import EmulatorType, DeviceTypeName
    HAS_CLOUD_SDK = True
    print('✓ Pasqal Cloud SDK available')
except ImportError:
    HAS_CLOUD_SDK = False
    print('⚠ Pasqal Cloud SDK not installed. Install with: pip install pasqal-cloud')

# Pulser for sequence design
try:
    from pulser import Pulse, Sequence, Register
    from pulser.devices import DigitalAnalogDevice
    from pulser.waveforms import RampWaveform, ConstantWaveform
    HAS_PULSER = True
    print('✓ Pulser SDK available')
except ImportError:
    HAS_PULSER = False
    print('⚠ Pulser not installed. Install with: pip install pulser')

# Check for real data
REAL_CSV = "../data/processed/optimized_sensor_network.csv"
HAS_REAL_DATA = os.path.exists(REAL_CSV)

if HAS_REAL_DATA:
    print(f'✓ Real sensor data found: {REAL_CSV}')
else:
    print(f'⚠ Real data not found, will use synthetic')

print('\n── Setup complete ──')

✓ Pasqal Cloud SDK available
✓ Pulser SDK available
✓ Real sensor data found: ../data/processed/optimized_sensor_network.csv

── Setup complete ──


---
## 1. Cloud Authentication

**Setup Instructions:**
1. Sign up at https://portal.pasqal.cloud/
2. Create or join a project
3. Get your project ID from the portal
4. Set environment variables (recommended) or use interactive login

**Security Best Practice:**
```bash
export PASQAL_USERNAME="your_email@example.com"
export PASQAL_PASSWORD="your_password"
export PASQAL_PROJECT_ID="your_project_id"
```

In [2]:
# =========================
# PASQAL CLOUD AUTH (FIXED CHECK)
# =========================
from pasqal_cloud import SDK

USERNAME = "mmei4@hawk.illinoistech.edu"
PROJECT_ID = "235a327e-bc98-46fa-8917-460621780db4"

try:
    sdk = SDK(
        username=USERNAME,
        password=None,   # prompts for password
        project_id=PROJECT_ID
    )

    # Try a simple API call that SHOULD exist
    batches = sdk.get_batches()  # lightweight check

    print("✓ Logged into Pasqal Cloud!")
    # print(f"✓ Found {list(batches)} existing batches")
    batches = list(sdk.get_batches())
    print(len(batches)) #testing simple API call for authetnication, should print a number

    HAS_CLOUD_ACCESS = True

except Exception as e:
    print("✗ Authentication FAILED")
    print("Error:", e)

    HAS_CLOUD_ACCESS = False

Enter your password: ········


✓ Logged into Pasqal Cloud!
3


---
## 2. Instance Generation (Same as Before)

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """Distance in meters."""
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def generate_instance(N, K, output_path=None, use_real_data=True):
    """Generate instance from real CSV or synthetic."""
    
    if use_real_data and HAS_REAL_DATA:
        df = pd.read_csv(REAL_CSV)
        if len(df) < N:
            print(f'    CSV has only {len(df)} rows, using synthetic')
            use_real_data = False
        else:
            df = df.head(N).copy()
            lats = df['latitude'].values
            lons = df['longitude'].values
            
            if 'risk_score_norm' in df.columns:
                risk_norm = df['risk_score_norm'].values
            else:
                base = np.random.uniform(0.3, 0.7, N)
                risk_norm = base / base.max()
            
            sensor_ids = df['sensor_id'].values if 'sensor_id' in df.columns else [f'SENSOR_{i:02d}' for i in range(N)]
            layers = df['layer'].values
            risk_source = "REAL - from CSV"
    
    if not use_real_data or not HAS_REAL_DATA:
        np.random.seed(42)
        lats = np.random.uniform(19.03, 19.18, N)
        lons = np.random.uniform(-104.35, -104.15, N)
        base = np.random.uniform(0.2, 0.7, N)
        risk_norm = base / base.max()
        sensor_ids = [f'SENSOR_{i:02d}' for i in range(N)]
        layers = ['Asset_Defense'] * (N//2) + ['Wildland_Perimeter'] * (N - N//2)
        risk_source = "SYNTHETIC"
    
    # Distance matrix
    dist_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            d = haversine(lats[i], lons[i], lats[j], lons[j])
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d
    
    instance = {
        "metadata": {
            "instance_name": f"pasqal_cloud_N{N}_K{K}",
            "created_date": datetime.now().strftime("%Y-%m-%d"),
            "platform": "Pasqal Cloud",
            "risk_source": risk_source
        },
        "problem": {
            "N": N,
            "K_budget": K,
            "min_separation_m": 2000,
            "lambda_budget": 8.0,
            "lambda_spatial": 12.0
        },
        "candidates": [],
        "distance_matrix": dist_matrix.tolist()
    }
    
    for i in range(N):
        instance['candidates'].append({
            "id": i,
            "sensor_id": str(sensor_ids[i]),
            "latitude": float(lats[i]),
            "longitude": float(lons[i]),
            "layer": str(layers[i]),
            "risk_score_norm": float(risk_norm[i])
        })
    
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            json.dump(instance, f, indent=2)
        print(f'  Saved: {output_path}')
    
    print(f'  N={N}, K={K}, Source={risk_source}')
    return instance

print('✓ Instance generator defined')

✓ Instance generator defined


---
## 3. Coordinate Transformation & Register Creation

In [4]:
def latlon_to_cartesian(lats, lons):
    """Convert lat/lon to local Cartesian (meters)."""
    lat0, lon0 = np.mean(lats), np.mean(lons)
    m_per_deg_lat = 111320
    m_per_deg_lon = 111320 * np.cos(np.radians(lat0))
    x = (lons - lon0) * m_per_deg_lon
    y = (lats - lat0) * m_per_deg_lat
    return x, y

def scale_to_pulser_register(x_m, y_m, target_span_um=50.0):
    """Scale Cartesian (m) to Pulser register (μm)."""
    x_m = x_m - np.mean(x_m)
    y_m = y_m - np.mean(y_m)
    span_m = max(np.ptp(x_m), np.ptp(y_m))
    scale = target_span_um / (span_m * 1e6)
    x_um = x_m * 1e6 * scale
    y_um = y_m * 1e6 * scale
    m_per_um = span_m / target_span_um
    return x_um, y_um, m_per_um

def instance_to_register(instance, target_span_um=50.0):
    """Convert instance to Pulser Register."""
    lats = np.array([c['latitude'] for c in instance['candidates']])
    lons = np.array([c['longitude'] for c in instance['candidates']])
    x_m, y_m = latlon_to_cartesian(lats, lons)
    x_um, y_um, m_per_um = scale_to_pulser_register(x_m, y_m, target_span_um)
    
    coords = {f"q{i}": (x_um[i], y_um[i]) for i in range(len(lats))}
    register = Register(coords)
    
    min_sep = instance['problem']['min_separation_m']
    blockade_radius_um = min_sep / m_per_um
    
    print(f'  Register: {len(coords)} atoms')
    print(f'  Span: {np.ptp(x_um):.1f} × {np.ptp(y_um):.1f} μm')
    print(f'  Blockade radius ({min_sep}m): {blockade_radius_um:.2f} μm')
    
    return register, blockade_radius_um, m_per_um

print('✓ Coordinate transformation defined')

✓ Coordinate transformation defined


---
## 4. Cloud-Ready Pulse Sequence Builder

In [5]:
def build_cloud_sequence(instance, target_span_um=50.0, duration_us=3.0, max_rabi_MHz=2.0):
    """
    Build sequence with safe parameters that definitely work with DigitalAnalogDevice.
    Uses hardcoded safe values based on DigitalAnalogDevice specifications.
    """
    if not HAS_PULSER:
        raise ImportError('Pulser not installed')
    
    # Build register
    register, blockade_radius_um, m_per_um = instance_to_register(instance, target_span_um)
    
    # Device
    device = DigitalAnalogDevice
    
    # Create sequence
    seq = Sequence(register, device)
    seq.declare_channel('global', 'rydberg_global')
    
    # FIXED: Hardcoded safe values from DigitalAnalogDevice specs
    # DigitalAnalogDevice specs:
    #   max_amp = 62.83 rad/μs (~10 MHz)
    #   max_abs_detuning = 125.66 rad/μs (~20 MHz)
    max_rabi_rad_us = min(max_rabi_MHz * 2 * np.pi, 50.0)  # Cap at ~8 MHz
    max_detuning_rad_us = 100.0  # ~16 MHz, safely within 125.66 limit
    
    # Phase 1: Ramp up Rabi (20% of duration)
    ramp_duration = int(duration_us * 1000 * 0.2)
    omega_ramp = RampWaveform(ramp_duration, 0, max_rabi_rad_us)
    delta_initial = -max_detuning_rad_us
    delta_ramp = ConstantWaveform(ramp_duration, delta_initial)
    seq.add(Pulse(omega_ramp, delta_ramp, 0), 'global')
    
    # Phase 2: Main sweep - from negative to positive detuning (60% of duration)
    sweep_duration = int(duration_us * 1000 * 0.6)
    omega_const = ConstantWaveform(sweep_duration, max_rabi_rad_us)
    delta_final = max_detuning_rad_us * 0.3
    delta_sweep = RampWaveform(sweep_duration, delta_initial, delta_final)
    seq.add(Pulse(omega_const, delta_sweep, 0), 'global')
    
    # Phase 3: Ramp down Rabi (20% of duration)
    rampdown_duration = int(duration_us * 1000 * 0.2)
    omega_down = RampWaveform(rampdown_duration, max_rabi_rad_us, 0)
    delta_down = ConstantWaveform(rampdown_duration, delta_final)
    seq.add(Pulse(omega_down, delta_down, 0), 'global')
    
    print(f'  Sequence: {duration_us:.2f} μs, Ω_max={max_rabi_rad_us/(2*np.pi):.2f} MHz')
    print(f'  Detuning sweep: {delta_initial/(2*np.pi):.2f} → {delta_final/(2*np.pi):.2f} MHz')
    print(f'  Register: {len(register.qubits)} atoms')
    
    metadata = {
        'duration_us': duration_us,
        'max_rabi_MHz': max_rabi_MHz,
        'n_atoms': len(register.qubits)
    }
    
    return seq, metadata

---
## 5. Cloud Job Submission

**Job Parameters:**
- `runs`: Number of measurement shots (1000-10000)
- `device_type`: Target device (FRESNEL, EMU_FREE, EMU_TN, EMU_MPS)
- `tags`: Custom tags for organization
- `wait`: Whether to block until completion

In [6]:
def submit_to_cloud(instance, device_type='EMU_MPS', runs=1000, wait=True, tags=None):
    """Submit job to Pasqal Cloud with safe sequence parameters."""
    if not HAS_CLOUD_ACCESS:
        raise RuntimeError('Cloud access not configured. Check authentication.')
    
    print(f'\n═══ Submitting to Pasqal Cloud ═══')
    print(f'Device: {device_type}')
    print(f'Runs: {runs}')
    print(f'Instance: N={instance["problem"]["N"]}, K={instance["problem"]["K_budget"]}')
    
    # FIXED: Use safe sequence builder with appropriate parameters
    # Adjust duration based on atom count to avoid simulation timeouts
    n_atoms = instance["problem"]["N"]
    if n_atoms <= 10:
        duration_us = 3.0
    elif n_atoms <= 15:
        duration_us = 2.5
    else:
        duration_us = 2.0
    
    max_rabi_MHz = 2.0
    
    seq, metadata = build_cloud_sequence(instance, duration_us=duration_us, max_rabi_MHz=max_rabi_MHz)
    
    # Serialize for cloud
    serialized_seq = seq.to_abstract_repr()
    
    # FIXED: Proper device type mapping
    device_map = {
        'FRESNEL': DeviceTypeName.FRESNEL,
        'EMU_FREE': EmulatorType.EMU_FREE,
        'EMU_TN': EmulatorType.EMU_TN,
        'EMU_MPS': EmulatorType.EMU_MPS,
    }
    device_enum = device_map.get(device_type)
    
    if device_enum is None:
        raise ValueError(f'Unknown device type: {device_type}')
    
    # Create job
    job = {"runs": runs}
    
    if tags is None:
        tags = ["wildfire_sensors", f"N{instance['problem']['N']}", f"K{instance['problem']['K_budget']}"]
    
    try:
        print('\nSubmitting batch...')
        batch = sdk.create_batch(
            serialized_seq,
            [job],
            device_type=device_enum,
            tags=tags,
            wait=wait
        )
        
        print(f'✓ Batch ID: {batch.id}')
        print(f'  Status: {batch.status}')
        
        if wait:
            print('\n  Waiting for results...')
            # Force refresh to get updated status
            batch.refresh()
            print(f'  Final status: {batch.status}')
            
            for job_obj in batch.ordered_jobs:
                print(f'\n  Job {job_obj.id}:')
                print(f'    Status: {job_obj.status}')
                if hasattr(job_obj, 'result') and job_obj.result:
                    print(f'    Result available: {len(job_obj.result)} measurements')
        else:
            print('\n  Job submitted (not waiting for completion)')
            print(f'  Check status: batch.get_status() or batch.wait()')
        
        return batch
    
    except Exception as e:
        print(f'\n✗ Submission failed: {e}')
        raise

---
## 6. Result Processing

In [7]:
def process_cloud_results(batch, instance):
    """
    Extract solution from Pasqal Cloud results.
    
    Returns:
        result dict with coverage, violations, selected sensors
    """
    N = instance['problem']['N']
    K = instance['problem']['K_budget']
    risk = np.array([c['risk_score_norm'] for c in instance['candidates']])
    dist = np.array(instance['distance_matrix'])
    min_sep = instance['problem']['min_separation_m']
    
    # Get results from first job
    job = batch.ordered_jobs[0]
    
    if not job.result:
        return {'error': 'No result available', 'status': job.status}
    
    # Parse measurements
    # Pasqal returns dict: {bitstring: count}
    counts = Counter()
    for measurement in job.result:
        # measurement format depends on device
        # Typically: list of 0/1 or bitstring
        if isinstance(measurement, (list, tuple)):
            bitstring = ''.join(str(int(b)) for b in measurement)
        else:
            bitstring = str(measurement)
        counts[bitstring] += 1
    
    # Find best solution
    best_bitstring = None
    best_score = -np.inf
    
    for bitstring, count in counts.most_common(50):
        if len(bitstring) != N:
            continue
        
        x = np.array([int(b) for b in bitstring], dtype=int)
        coverage = float(np.sum(risk * x))
        
        # Penalties
        budget_penalty = abs(x.sum() - K) * 10.0 #Changed from 0.1 -> 10
        
        selected = np.where(x == 1)[0]
        viol = 0
        for i in selected:
            for j in selected:
                if i < j and 0 < dist[i, j] < min_sep:
                    viol += 1
        
        score = coverage - budget_penalty - viol * 0.05
        
        if score > best_score:
            best_score = score
            best_bitstring = bitstring
    
    if best_bitstring is None:
        return {'error': 'No valid solution found'}
    
    # Extract final solution
    x_best = np.array([int(b) for b in best_bitstring], dtype=int)
    selected = np.where(x_best == 1)[0]
    coverage = float(np.sum(risk[selected]))
    
    viol = 0
    for i in selected:
        for j in selected:
            if i < j and 0 < dist[i, j] < min_sep:
                viol += 1
    
    return {
        'coverage': coverage,
        'violations': int(viol),
        'n_selected': int(x_best.sum()),
        'selected_indices': selected.tolist(),
        'bitstring': best_bitstring,
        'total_counts': sum(counts.values()),
        'unique_bitstrings': len(counts),
        'top_5_counts': dict(counts.most_common(5))
    }

print('✓ Result processing defined')

✓ Result processing defined


---
## Benchmarking: Classical Baselines

Before running on Pasqal Cloud, we need classical baselines for comparison:
- **Greedy:** Sort by risk, take top-K
- **Simulated Annealing:** Metropolis-Hastings optimization

These will let us measure quantum advantage (or lack thereof).

In [8]:
# ── Classical Baselines ────────────────────────────────────────────────────────

def build_qubo(instance):
    """Build Q matrix from instance."""
    N = instance['problem']['N']
    K = instance['problem']['K_budget']
    lam_b = instance['problem']['lambda_budget']
    lam_s = instance['problem']['lambda_spatial']
    min_sep = instance['problem']['min_separation_m']
    
    r = np.array([c['risk_score_norm'] for c in instance['candidates']])
    dist = np.array(instance['distance_matrix'])
    
    Q = np.zeros((N, N))
    for i in range(N):
        Q[i, i] = -r[i] + lam_b * (1 - 2*K)
    for i in range(N):
        for j in range(i+1, N):
            Q[i, j] = 2 * lam_b
            if 0 < dist[i, j] < min_sep:
                Q[i, j] += 2 * lam_s
    
    return Q, r, dist

def qubo_energy(x, Q):
    return float(x @ Q @ x)

def count_violations(x, dist, min_sep):
    selected = np.where(x == 1)[0]
    viol = 0
    for i in selected:
        for j in selected:
            if i < j and 0 < dist[i, j] < min_sep:
                viol += 1
    return viol

def solve_greedy(instance):
    Q, r, dist = build_qubo(instance)
    K = instance['problem']['K_budget']
    
    t0 = time.time()
    indices = np.argsort(r)[::-1][:K]
    x = np.zeros(len(r), dtype=int)
    x[indices] = 1
    runtime = time.time() - t0
    
    return {
        'coverage': float(r[indices].sum()),
        'energy': float(qubo_energy(x, Q)),
        'violations': int(count_violations(x, dist, instance['problem']['min_separation_m'])),
        'runtime': runtime,
        'n_selected': int(x.sum()),
        'selected_indices': indices.tolist()
    }

def solve_sa(instance, seed=None, n_steps=50000):
    if seed is not None:
        np.random.seed(seed)
    
    Q, r, dist = build_qubo(instance)
    N = len(r)
    
    t0 = time.time()
    x = np.random.randint(0, 2, N)
    E = qubo_energy(x, Q)
    best_x, best_E = x.copy(), E
    
    temps = np.linspace(5.0, 0.01, n_steps)
    for T in temps:
        flip = np.random.randint(N)
        x_new = x.copy()
        x_new[flip] = 1 - x_new[flip]
        E_new = qubo_energy(x_new, Q)
        dE = E_new - E
        if dE < 0 or np.random.rand() < np.exp(-dE / T):
            x, E = x_new, E_new
            if E < best_E:
                best_x, best_E = x.copy(), E
    
    runtime = time.time() - t0
    selected = np.where(best_x == 1)[0]
    
    return {
        'coverage': float(r[selected].sum() if len(selected) > 0 else 0),
        'energy': float(best_E),
        'violations': int(count_violations(best_x, dist, instance['problem']['min_separation_m'])),
        'runtime': runtime,
        'n_selected': int(best_x.sum()),
        'selected_indices': selected.tolist()
    }

print('✓ Classical solvers defined')

✓ Classical solvers defined


---
## 📊 Comprehensive Benchmark: Pasqal vs Classical

**Test Plan:**
1. Generate instances: N=8, 10, 12 (within EMU_FREE limits)
2. Run each solver with multiple seeds/configurations
3. Measure: coverage, violations, runtime, variance
4. Build comparison table and visualization

**Solvers:**
- **Greedy** (deterministic baseline)
- **Simulated Annealing** (classical optimization, 3 seeds)
- **Pasqal Cloud** (quantum/emulator, varying shot counts)

In [9]:
# Benchmark Configuration
BENCHMARK_CONFIG = {
    'instances': [
        {'N': 8, 'K': 3},
        {'N': 10, 'K': 4},
        {'N':17, 'K': 8},
        {'N':25, 'K':8},
    ],
    'seeds': [42, 123, 789],
    'sa_steps': 50000,
    'pasqal_runs': [500, 1000],
    'pasqal_device': 'EMU_MPS',
}

print('Benchmark config loaded')
print(f"  Instances: {BENCHMARK_CONFIG['instances']}")
print(f"  Seeds: {BENCHMARK_CONFIG['seeds']}")

Benchmark config loaded
  Instances: [{'N': 8, 'K': 3}, {'N': 10, 'K': 4}, {'N': 17, 'K': 8}, {'N': 25, 'K': 8}]
  Seeds: [42, 123, 789]


In [10]:
# Run Comprehensive Benchmark
benchmark_results = []

for config in BENCHMARK_CONFIG['instances']:
    N, K = config['N'], config['K']
    print(f'\n{"="*60}')
    print(f'BENCHMARK: N={N}, K={K}')
    print(f'{"="*60}')
    
    # Generate instance
    instance_path = f'cloud_results/benchmark/N{N}_K{K}.json'
    print('\n1. Generating instance...')
    instance = generate_instance(N, K, output_path=instance_path)
    
    result = {'N': N, 'K': K, 'solvers': {}}
    
    # Greedy
    print('\n2. Greedy...')
    res_greedy = solve_greedy(instance)
    print(f"   Coverage: {res_greedy['coverage']:.4f}, Violations: {res_greedy['violations']}")
    result['solvers']['greedy'] = res_greedy
    
    # Simulated Annealing
    print(f'\n3. SA (seeds={BENCHMARK_CONFIG["seeds"]})...')
    sa_runs = []
    for seed in BENCHMARK_CONFIG['seeds']:
        res = solve_sa(instance, seed=seed, n_steps=BENCHMARK_CONFIG['sa_steps'])
        sa_runs.append(res)
        print(f"   Seed {seed}: coverage={res['coverage']:.4f}, violations={res['violations']}")
    
    sa_cov = [r['coverage'] for r in sa_runs]
    sa_viol = [r['violations'] for r in sa_runs]
    result['solvers']['sa'] = {
        'mean_coverage': float(np.mean(sa_cov)),
        'std_coverage': float(np.std(sa_cov)),
        'mean_violations': float(np.mean(sa_viol)),
        'all_runs': sa_runs
    }
    print(f"   Summary: {np.mean(sa_cov):.4f}±{np.std(sa_cov):.4f}")
    
    # Pasqal Cloud
    if HAS_CLOUD_ACCESS:
        print(f'\n4. Pasqal ({BENCHMARK_CONFIG["pasqal_device"]})...')
        pasqal_runs = []
        for runs in BENCHMARK_CONFIG['pasqal_runs']:
            print(f"   Runs={runs}...", end=' ')
            try:
                batch = submit_to_cloud(
                    instance,
                    device_type=BENCHMARK_CONFIG['pasqal_device'],
                    runs=runs,
                    wait=True,
                    tags=[f'N{N}K{K}_runs{runs}']
                )
                res = process_cloud_results(batch, instance)
                res['runs'] = runs
                res['batch_id'] = batch.id
                pasqal_runs.append(res)
                if 'error' not in res:
                    print(f"coverage={res['coverage']:.4f}, violations={res['violations']}")
                else:
                    print(f"ERROR: {res['error']}")
            except Exception as e:
                print(f"FAILED: {e}")
        
        valid = [r for r in pasqal_runs if 'error' not in r]
        if valid:
            pasqal_cov = [r['coverage'] for r in valid]
            pasqal_viol = [r['violations'] for r in valid]
            result['solvers']['pasqal'] = {
                'mean_coverage': float(np.mean(pasqal_cov)),
                'std_coverage': float(np.std(pasqal_cov)),
                'mean_violations': float(np.mean(pasqal_viol)),
                'all_runs': pasqal_runs
            }
            print(f"   Summary: {np.mean(pasqal_cov):.4f}±{np.std(pasqal_cov):.4f}")
    else:
        print('\n4. Pasqal: SKIPPED (not authenticated)')
        result['solvers']['pasqal'] = {'error': 'Not authenticated'}
    
    benchmark_results.append(result)

# Save results
os.makedirs('cloud_results/benchmark', exist_ok=True)
result_path = f"cloud_results/benchmark/results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(result_path, 'w') as f:
    json.dump(benchmark_results, f, indent=2)

print(f'\n{"="*60}')
print(f'✓ Benchmark complete: {result_path}')
print(f'{"="*60}')


BENCHMARK: N=8, K=3

1. Generating instance...
  Saved: cloud_results/benchmark/N8_K3.json
  N=8, K=3, Source=REAL - from CSV

2. Greedy...
   Coverage: 2.5435, Violations: 1

3. SA (seeds=[42, 123, 789])...
   Seed 42: coverage=2.5297, violations=0
   Seed 123: coverage=2.5297, violations=0
   Seed 789: coverage=2.5297, violations=0
   Summary: 2.5297±0.0000

4. Pasqal (FRESNEL)...
   Runs=500... 
═══ Submitting to Pasqal Cloud ═══
Device: FRESNEL
Runs: 500
Instance: N=8, K=3
  Register: 8 atoms
  Span: 29.6 × 50.0 μm
  Blockade radius (2000m): 10.49 μm
  Sequence: 3.00 μs, Ω_max=2.00 MHz
  Detuning sweep: -15.92 → 4.77 MHz
  Register: 8 atoms

Submitting batch...


BatchCreationError: Batch creation failed: CB1107: You cannot create a batch with this Device type on this project.
Details: {
  "status": "fail",
  "message": "Bad request.",
  "code": "CB1107",
  "data": {
    "description": "You cannot create a batch with this Device type on this project."
  }
}

---
## 📋 Results Table & Visualization

In [ ]:
# Display Results Table
if benchmark_results:
    print('\n' + '='*90)
    print('BENCHMARK RESULTS')
    print('='*90 + '\n')
    
    print(f'{"Instance":<12} {"Solver":<12} {"Coverage":<20} {"Violations":<15} {"Runtime":<10}')
    print('-'*90)
    
    for res in benchmark_results:
        N, K = res['N'], res['K']
        instance_label = f'N={N}, K={K}'
        
        # Greedy
        g = res['solvers']['greedy']
        print(f'{instance_label:<12} {"Greedy":<12} {g["coverage"]:>6.4f}{"":<14} {g["violations"]:>4d}{"":<11} {g["runtime"]:>8.4f}s')
        
        # SA
        sa = res['solvers']['sa']
        cov_str = f"{sa['mean_coverage']:.4f}±{sa['std_coverage']:.4f}"
        viol_str = f"{sa['mean_violations']:.2f}±{sa.get('std_violations', 0):.2f}"
        print(f'{"":<12} {"SA (mean)":<12} {cov_str:<20} {viol_str:<15} {"":<10}')
        
        # Pasqal
        if 'pasqal' in res['solvers'] and 'error' not in res['solvers']['pasqal']:
            p = res['solvers']['pasqal']
            cov_str = f"{p['mean_coverage']:.4f}±{p['std_coverage']:.4f}"
            viol_str = f"{p['mean_violations']:.2f}±{p.get('std_violations', 0):.2f}"
            print(f'{"":<12} {"Pasqal":<12} {cov_str:<20} {viol_str:<15} {"":<10}')
        elif 'pasqal' in res['solvers']:
            print(f'{"":<12} {"Pasqal":<12} {res["solvers"]["pasqal"]["error"]:<20}')
        
        print('-'*90)
    
    print('\n✓ Table complete')
else:
    print('⚠ No benchmark results available. Run benchmark first.')

In [ ]:
# Visualization: Coverage Comparison
if benchmark_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Coverage by instance
    ax = axes[0]
    x_labels = [f"N={r['N']}" for r in benchmark_results]
    x_pos = np.arange(len(x_labels))
    width = 0.25
    
    greedy_cov = [r['solvers']['greedy']['coverage'] for r in benchmark_results]
    sa_cov = [r['solvers']['sa']['mean_coverage'] for r in benchmark_results]
    sa_std = [r['solvers']['sa']['std_coverage'] for r in benchmark_results]
    
    ax.bar(x_pos - width, greedy_cov, width, label='Greedy', color='#ff7f0e')
    ax.bar(x_pos, sa_cov, width, yerr=sa_std, label='SA', color='#2ca02c', capsize=5)
    
    # Add Pasqal if available
    pasqal_available = all('pasqal' in r['solvers'] and 'error' not in r['solvers']['pasqal'] for r in benchmark_results)
    if pasqal_available:
        pasqal_cov = [r['solvers']['pasqal']['mean_coverage'] for r in benchmark_results]
        pasqal_std = [r['solvers']['pasqal']['std_coverage'] for r in benchmark_results]
        ax.bar(x_pos + width, pasqal_cov, width, yerr=pasqal_std, label='Pasqal', color='#1f77b4', capsize=5)
    
    ax.set_xlabel('Instance Size', fontsize=12)
    ax.set_ylabel('Coverage (Normalized Risk Sum)', fontsize=12)
    ax.set_title('Coverage Comparison', fontsize=14, weight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    # Plot 2: Violations
    ax = axes[1]
    greedy_viol = [r['solvers']['greedy']['violations'] for r in benchmark_results]
    sa_viol = [r['solvers']['sa']['mean_violations'] for r in benchmark_results]
    
    ax.bar(x_pos - width, greedy_viol, width, label='Greedy', color='#ff7f0e')
    ax.bar(x_pos, sa_viol, width, label='SA', color='#2ca02c')
    
    if pasqal_available:
        pasqal_viol = [r['solvers']['pasqal']['mean_violations'] for r in benchmark_results]
        ax.bar(x_pos + width, pasqal_viol, width, label='Pasqal', color='#1f77b4')
    
    ax.set_xlabel('Instance Size', fontsize=12)
    ax.set_ylabel('Proximity Violations (<500m)', fontsize=12)
    ax.set_title('Constraint Violations', fontsize=14, weight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cloud_results/benchmark/comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\n✓ Visualization saved: cloud_results/benchmark/comparison.png')
else:
    print('⚠ No results to visualize')

---
## 8. Advanced: Submit to Fresnel QPU (Hardware)

**Requirements:**
- Approved quota for Fresnel access
- N ≤ 100 (current Fresnel limit)
- May incur billing charges

**Uncomment and run after quota approval:**

In [ ]:
# # Generate larger instance for hardware
# instance_hardware = generate_instance(N=20, K=8, output_path='cloud_results/instances/fresnel_N20_K8.json')

# # Submit to Fresnel QPU
# if HAS_CLOUD_ACCESS:
#     batch_hardware = submit_to_cloud(
#         instance_hardware,
#         device_type='FRESNEL',  # Real quantum hardware!
#         runs=5000,  # More shots for hardware noise
#         wait=True,
#       
#     )
#     
#     result_hardware = process_cloud_results(batch_hardware, instance_hardware)
#     print(f"\nFresnel QPU Result:")
#     print(f"  Coverage: {result_hardware['coverage']:.4f}")
#     print(f"  Violations: {result_hardware['violations']}")

print('\n⚠ Fresnel submission commented out (requires quota approval)')
print('  Uncomment code above after getting hardware access')

---
## 10. Key Takeaways & Next Steps

### What You Just Did
 Authenticated with Pasqal Cloud  
 Created atom register from real sensor coordinates  
 Built adiabatic optimization pulse sequence  
 Submitted job to cloud (emulator or QPU)  
 Retrieved and processed quantum measurement results  

### Device Comparison
| Device | Qubits | Speed | Cost | Use Case |
|--------|--------|-------|------|----------|
| **EMU_FREE** | ≤10 | Slow | Free | Testing, small demos |
| **EMU_TN** | 10-100 | Medium | Paid | Development, N=20-50 |
| **EMU_MPS** | 10-100 | Fast (GPU) | Paid | Production simulations |
| **FRESNEL** | ≤100 | Fast | Paid | Real quantum advantage |

### Pasqal Cloud Resources
- Portal: https://portal.pasqal.cloud/
- Docs: https://docs.pasqal.com/cloud/
- Pricing: https://portal.pasqal.cloud/offers
- Support: https://portal.pasqal.cloud/contact

---

In [ ]:
if HAS_CLOUD_ACCESS:
    print("\n═══ Available Devices for Your Project ═══")
    
    # --- 1. Get Device Specifications (The correct way) ---
    try:
        # This is the confirmed method name in the SDK
        device_specs_dict = sdk.get_device_specs_dict()
        
        if device_specs_dict:
            print("  Successfully retrieved device specifications.")
            print(f"  Available device types: {list(device_specs_dict.keys())}")
        else:
            print("  No device specifications found.")
            
    except Exception as e:
        print(f"  Error getting device specs: {e}")
        print("  This might indicate a network issue or a problem with your project's permissions.")

    # --- 2. Get All Your Projects (The correct way) ---
    print("\n═══ Your Accessible Projects ═══")
    try:
        # This method is documented to return a list of Project objects
        all_projects = sdk.get_all_projects()
        
        if all_projects:
            print(f"  Found {len(all_projects)} project(s):")
            for project in all_projects:
                # Project objects have an 'id' attribute
                print(f"    - Project ID: {project.id}")
                # You can also print other attributes if needed, e.g., project.name
        else:
            print("  No projects found for this user.")
            
    except Exception as e:
        print(f"  Error getting projects: {e}")
        print("  This could be a permissions issue. Ensure your account is active and has projects.")